# Analyse games with KataGo on Modal

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SoumyaK4/SWHub/blob/SWHub/Docs/Free%20Cloud%20Katago/Analysis/Kata_Modal.ipynb)

Use the **control panel** below. You can choose settings, start the server, copy its connection address and stop it without editing code.

During setup, the panel shows the current step, elapsed time and three recent log lines. If a step is quiet, it shows the time since the last log. Full logs remain below.

## Open the panel

1. Open this notebook in **Colab with a CPU runtime**, or in local Jupyter. Modal supplies the T4 GPU. Have a Modal account with a valid payment method, even when using free credits.
2. Click **▶ on the Open control panel block directly below this guide**. Wait for the panel to appear in that block. This is the only code block you need to run. It prepares the controls without logging in, starting a server, deploying an app or running Stop.
3. In the panel, click **1. Login**, open the printed browser link and authorize your intended workspace. Choose it from the dropdown, leave Environment as `main`, then click **2. Select workspace**. The selection is checked without starting a GPU.
4. Keep Medium transformer and HumanSL selected, then click **3. Deploy**. Wait for the action to finish.
5. Copy the complete address in **SWHub URL** into **Tools > KataGo > Settings > Remote**, then select **Save**. Use the status dot or **Retry connection** if needed.
6. When finished, click **Stop Modal app** in this panel.

Keep the connection address private. You can close Colab after deployment. Closing the notebook does not stop the deployed Modal app. Stopping keeps cached downloads. To use it again, click Deploy.

All code blocks are collapsed by default. Expand them only if you want to inspect the implementation. If controls do not appear, reconnect and run the top Open control panel block again. Trust the notebook in local Jupyter. After changing settings, stop an active Colab server first, then start again. Modal changes take effect after Deploy.


In [ ]:
# @title ▶ Open control panel
# Generated by tool/katago_notebooks/sync_controls.py.
# Implementation sources remain readable in the collapsed cells below.
%pip install -q modal==1.5.5 "ipywidgets>=7.7.1,<9"

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

"""Shared controls embedded in all four standalone KataGo notebooks.

Run sync_controls.py after editing. No repository checkout is needed by users.
"""

import ast
import builtins
from collections import deque
import html
import json
import re
import subprocess
import threading
import time
from pathlib import Path

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import display


# Registering a cell saves its source without running setup or starting a GPU.
SWHUB_ACTIONS = {}


def swhub_action(line, cell):
    SWHUB_ACTIONS[line.strip()] = cell


get_ipython().register_magic_function(swhub_action, "cell", "swhub_action")


def swhub_configure_source(source, settings):
    """Replace only top-level setting assignments, preserving embedded servers."""
    tree = ast.parse(source)
    lines = source.splitlines(keepends=True)
    for node in reversed(tree.body):
        if not isinstance(node, ast.Assign) or len(node.targets) != 1:
            continue
        target = node.targets[0]
        if isinstance(target, ast.Name) and target.id in settings:
            lines[node.lineno - 1:node.end_lineno] = [
                f"{target.id} = {settings[target.id]!r}\n"
            ]
    return "".join(lines)


def swhub_run_action(name, settings=None, on_output=None):
    source = SWHUB_ACTIONS[name]
    # Existing writefile cells remain readable, self-contained service sources.
    lines = source.splitlines(keepends=True)
    writefile = next((i for i, line in enumerate(lines)
                      if line.startswith("%%writefile ")), None)
    if writefile is not None:
        path = Path(lines[writefile].split(maxsplit=1)[1].strip())
        source = swhub_configure_source("".join(lines[writefile + 1:]), settings or {})
        compile(source, str(path), "exec")
        path.write_text(source, encoding="utf-8")
        print("Prepared", path.name)
        if on_output is not None:
            on_output("Prepared " + path.name)
        return
    source = swhub_configure_source(source, settings or {})
    # Observe Python action logs without redirecting the kernel's stdout or
    # moving Drive authorization and IPython work off the calling thread.
    had_print = "print" in globals()
    original_print = globals().get("print", builtins.print)

    def report(*args, **kwargs):
        original_print(*args, **kwargs)
        if on_output is not None and kwargs.get("file") is None:
            separator = kwargs.get("sep", " ")
            on_output((" " if separator is None else separator).join(str(arg) for arg in args))

    globals()["print"] = report
    try:
        exec(compile(source, f"KataGo action: {name}", "exec"), globals())
    finally:
        if had_print:
            globals()["print"] = original_print
        else:
            globals().pop("print", None)


class SWHubKataGoPanel:
    def __init__(self, kind):
        self.kind = kind
        self.modal = kind.startswith("modal")
        self.contribution = kind.endswith("contribution")
        self.busy = False
        self.target = None
        self.activity_stop = threading.Event()
        self.activity_lock = threading.RLock()
        self.recent_activity = deque(maxlen=3)
        self.activity_secrets = ()
        self.buttons = {}
        self.fields = {}
        self.status = widgets.HTML(value="Ready. Choose your settings, then start below.")
        self.activity = widgets.HTML(layout=widgets.Layout(max_height="120px", overflow="auto"))
        self.endpoint = widgets.Text(description="SWHub URL", placeholder="Address appears after starting",
                                     layout=widgets.Layout(width="100%"))
        self.connection = widgets.HTML()
        self.output = widgets.Output(layout=widgets.Layout(
            max_height="180px", overflow="auto", border="1px solid #888"))
        self.progress = widgets.IntProgress(min=0, max=1, description="Progress")
        children = [widgets.HTML(value="<h2>SWHub · KataGo " +
                    ("Contribution" if self.contribution else "Analysis") + "</h2>")]
        if self.modal:
            children += [widgets.HTML(value="Modal supplies the T4 GPU. GPU use requires a valid "
                         "payment method. Closing this notebook does not stop the deployed app."),
                         self.button("login", "1. Login", self.login),
                         self.field("MODAL_PROFILE", widgets.Dropdown, "Workspace", options=[]),
                         self.field("MODAL_ENVIRONMENT", widgets.Text, "Environment", value="main"),
                         self.button("profiles", "Refresh workspaces", self.refresh_profiles),
                         self.button("select", "2. Select workspace", self.select_workspace)]
            self.fields["MODAL_PROFILE"].observe(self.invalidate_target, names="value")
            self.fields["MODAL_ENVIRONMENT"].observe(self.invalidate_target, names="value")
        if not self.contribution:
            children += [self.field("MAIN_MODEL_PRESET", widgets.Dropdown, "Model", options=[
                ("Medium transformer (recommended)", "transformer_medium"),
                ("Small transformer", "transformer_small"),
                ("Large transformer", "transformer_large"),
                ("Convolutional b18", "colab_fast_b18"),
                ("Convolutional b28", "latest_b28"),
                ("Convolutional b40 Zhizi", "strongest_b40_zhizi"),
                ("Custom model", "custom")]),
                self.field("ENABLE_HUMAN_MODEL", widgets.Checkbox, "HumanSL", value=True),
                widgets.HTML(value="Keep HumanSL enabled for Career, Subjects and Game Focus.")]
            custom = [self.field(key, widgets.Text, label) for key, label in (
                ("MODEL_URL", "Model URL"), ("MODEL_FILENAME", "Filename"),
                ("MODEL_SHA256", "SHA-256"))]
            advanced = widgets.Accordion(children=[widgets.VBox(custom)], selected_index=None)
            advanced.set_title(0, "Custom model settings (only for Custom model)")
            children.append(advanced)
        else:
            children.append(widgets.HTML(value="The training server chooses the model. Enter your "
                            "katagotraining.org account in SWHub. Finish games & stop in SWHub "
                            "before stopping this server. Sessions last up to four hours."))
        if self.kind == "colab_analysis":
            children += [self.field("TUNNEL_PROVIDER", widgets.Dropdown, "Connection", options=[
                ("Automatic (recommended)", "auto"), ("Cloudflare", "cloudflare"),
                ("Pinggy", "pinggy"), ("localhost.run", "localhostrun"),
                ("Local only", "none")]),
                self.field("USE_GOOGLE_DRIVE_CACHE", widgets.Checkbox, "Drive cache", value=True)]
        if self.kind == "modal_contribution":
            children += [self.field("ENABLE_CONTRIBUTOR_LOGIN", widgets.Checkbox,
                                    "Controller login", value=False),
                         self.field("CONTRIBUTOR_LOGIN_USERNAME", widgets.Text, "Username"),
                         self.field("CONTRIBUTOR_LOGIN_PASSWORD", widgets.Password, "Password"),
                         widgets.HTML(value="Optional extra server login, separate from your training account. "
                                      "Enter it again for each deployment. Keep the server URL private.")]
        children += [self.button("start", "3. Deploy" if self.modal else "Start server",
                                 self.start, "success"), self.progress, self.status, self.activity,
                     widgets.HTML(value="<b>Connect SWHub</b>. Copy the full address below into " +
                                  ("Tools → KataGo Contribution → Settings." if self.contribution else
                                   "Tools → KataGo → Settings → Remote, then select Save.")),
                     self.endpoint, self.connection,
                     self.button("logs", "Show recent logs", self.logs),
                     self.button("stop", "Stop Modal app" if self.modal else "Stop server",
                                 self.stop, "danger")]
        children += [widgets.HTML(value="<b>Full setup and troubleshooting logs</b>. "
                                  "Wait for an action to finish before clicking another. "
                                  "Use the notebook's Interrupt execution control if setup stalls."), self.output]
        self.view = widgets.VBox(children, layout=widgets.Layout(max_width="850px"))
        self.update_buttons()

    def field(self, key, widget_type, description, **kwargs):
        field = widget_type(description=description, style={"description_width": "120px"},
                            layout=widgets.Layout(width="100%"), **kwargs)
        self.fields[key] = field
        return field

    def button(self, key, description, action, style=""):
        button = widgets.Button(description=description, button_style=style,
                                layout=widgets.Layout(width="230px"))
        button.on_click(lambda _: self.perform(description, action))
        self.buttons[key] = button
        return button

    def update_buttons(self):
        for button in self.buttons.values():
            button.disabled = self.busy
        for field in self.fields.values():
            field.disabled = self.busy
        if self.modal and not self.busy:
            for key in ("start", "logs", "stop"):
                self.buttons[key].disabled = self.target is None
            self.buttons["select"].disabled = not self.fields["MODAL_PROFILE"].value

    def invalidate_target(self, change=None):
        self.target = None
        globals()["selected_modal_target"] = None
        self.endpoint.value = ""
        self.connection.value = ""
        self.status.value = "Select workspace to verify the new deployment target."
        self.update_buttons()

    def perform(self, label, action):
        if self.busy:
            return
        self.busy = True
        self.update_buttons()
        self.status.value = html.escape(label) + "…"
        self.progress.bar_style = "info"
        self.progress.max = 1
        self.progress.value = 0
        self.started_at = self.last_activity_at = time.monotonic()
        self.last_render_at = 0
        self.recent_activity.clear()
        secret = self.fields.get("CONTRIBUTOR_LOGIN_PASSWORD")
        self.activity_secrets = (secret.value,) if secret is not None and secret.value else ()
        self.activity_stop = threading.Event()
        self.render_activity()
        self.activity_thread = threading.Thread(target=self.watch_activity, daemon=True)
        self.activity_thread.start()
        with self.output:
            print("\n" + label)
            try:
                action()
            except (Exception, KeyboardInterrupt) as exc:
                self.progress.bar_style = "danger"
                self.status.value = "Action interrupted." if isinstance(exc, KeyboardInterrupt) else (
                    "Action failed. Read the details below, correct the problem and retry.")
                # Passwords are never printed in tracebacks or saved to notebook source.
                message = str(exc)
                for secret in self.activity_secrets:
                    message = message.replace(secret, "[redacted]")
                print(type(exc).__name__ + ": " + message)
            else:
                self.progress.bar_style = "success"
                self.progress.value = self.progress.max
                self.status.value = ("Server address ready. Connect SWHub below." if action == self.start
                                     else html.escape(label) + " complete.")
            finally:
                self.activity_stop.set()
                self.activity_thread.join(timeout=2)
                self.render_activity(finished=True)
                if action == self.start:
                    self.clear_controller_password()
                self.activity_secrets = ()
                self.busy = False
                self.update_buttons()

    def run_action(self, name, settings=None):
        swhub_run_action(name, settings, on_output=self.record_activity)

    def record_activity(self, message):
        with self.activity_lock:
            if not self.busy or self.activity_stop.is_set():
                return
            # Keep the preview bounded, strip terminal escapes, and escape HTML.
            message = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", message)
            for secret in self.activity_secrets:
                message = message.replace(secret, "[redacted]")
            for line in message.replace("\r", "\n").splitlines()[-3:]:
                line = " ".join(line.split())
                if not line:
                    continue
                self.last_activity_at = time.monotonic()
                elapsed = int(self.last_activity_at - self.started_at)
                self.recent_activity.append((elapsed, line[:180]))
                stage = re.match(r"\[(\d+)/(\d+)\]", line)
                if stage:
                    self.status.value = html.escape(line[:180])
                    self.progress.max = max(1, int(stage[2]))
                    self.progress.value = min(self.progress.max, max(0, int(stage[1]) - 1))
            if time.monotonic() - self.last_render_at >= 0.2:
                self.render_activity()

    def render_activity(self, finished=False):
        with self.activity_lock:
            now = time.monotonic()
            self.last_render_at = now
            elapsed = int(now - self.started_at)
            quiet = int(now - self.last_activity_at)
            timing = f"Elapsed {elapsed // 60}:{elapsed % 60:02d}"
            if not finished:
                timing += (f" · No new log for {quiet}s. Waiting for this step to finish."
                           if quiet >= 10 else " · Working")
            lines = [f"<div><small>{timing}</small></div>"]
            for stamp, line in self.recent_activity:
                lines.append(f"<div>{stamp // 60}:{stamp % 60:02d} · {html.escape(line)}</div>")
            self.activity.value = '<div style="overflow-wrap:anywhere">' + "".join(lines) + "</div>"

    def watch_activity(self):
        while not self.activity_stop.wait(1):
            self.render_activity()

    def refresh_profiles(self):
        self.invalidate_target()
        # Reject token overrides before interpreting CLI output as profiles.
        import os
        if os.environ.get("MODAL_TOKEN_ID") or os.environ.get("MODAL_TOKEN_SECRET"):
            raise RuntimeError("Token environment variables override workspace profiles. "
                               "Remove them or use a clean runtime, then Login again.")
        result = subprocess.run(["modal", "profile", "list", "--json"],
                                check=True, text=True, capture_output=True, timeout=60)
        profiles = json.loads(result.stdout)
        names = [row["name"] for row in profiles]
        self.fields["MODAL_PROFILE"].options = names
        print("Choose a workspace and select it." if names else "No saved workspaces. Login first.")

    def login(self):
        self.invalidate_target()
        self.run_action("login")
        self.refresh_profiles()

    def select_workspace(self):
        self.invalidate_target()
        self.run_action("workspace", {key: self.fields[key].value for key in (
            "MODAL_PROFILE", "MODAL_ENVIRONMENT")})
        self.target = globals()["selected_modal_target"]

    def settings(self):
        values = {key: field.value for key, field in self.fields.items()}
        if not self.contribution:
            values["HUMAN_MODEL_PRESET"] = (
                "official_human_sl_v0" if values["ENABLE_HUMAN_MODEL"] else "disabled")
            if values["MAIN_MODEL_PRESET"] == "custom":
                from urllib.parse import urlparse
                import re
                if urlparse(values["MODEL_URL"]).scheme != "https":
                    raise ValueError("Enter an HTTPS URL for the custom model.")
                filename = values["MODEL_FILENAME"]
                if not filename or filename in (".", "..") or "/" in filename or "\\" in filename:
                    raise ValueError("Enter a model filename without folders.")
                if re.fullmatch(r"[0-9a-fA-F]{64}", values["MODEL_SHA256"]) is None:
                    raise ValueError("Enter the custom model's 64-character SHA-256.")
            else:
                for key in ("MODEL_URL", "MODEL_FILENAME", "MODEL_SHA256"):
                    values.pop(key)
        if self.modal:
            values["KEEP_STREAMING_AFTER_READY"] = False
        return values

    def start(self):
        if not self.modal:
            for name in (("service", "tunnel") if self.contribution else ("ws_proc", "tunnel_proc")):
                process = globals().get(name)
                if process is not None and process.poll() is None:
                    raise RuntimeError("Server is already running. Stop server before starting again.")
        self.endpoint.value = ""
        self.connection.value = ""
        for key in ("final_wss_url", "final_status_url", "websocket_url", "base", "ready"):
            globals().pop(key, None)
        settings = self.settings()
        if self.modal and (self.target is None or self.target != globals().get("selected_modal_target")):
            self.invalidate_target()
            raise RuntimeError("Select workspace again before deploying.")
        if self.kind == "modal_contribution":
            if settings["ENABLE_CONTRIBUTOR_LOGIN"] and not all(settings[key].strip() for key in (
                    "CONTRIBUTOR_LOGIN_USERNAME", "CONTRIBUTOR_LOGIN_PASSWORD")):
                raise ValueError("Enter both controller login fields or disable Controller login.")
            globals().update({key: settings[key] for key in (
                "ENABLE_CONTRIBUTOR_LOGIN", "CONTRIBUTOR_LOGIN_USERNAME", "CONTRIBUTOR_LOGIN_PASSWORD")})
        steps = (["write_app", "deploy"] if self.modal else
                 ["prepare", "drive", "write_app", "start"] if self.contribution else ["start"])
        self.progress.max = len(steps)
        self.progress.value = 0
        try:
            for index, step in enumerate(steps):
                stage = {"prepare": "Checking GPU and installing dependencies",
                         "drive": "Connecting Google Drive and restoring cached files",
                         "write_app": "Preparing the server",
                         "deploy": "Deploying to Modal, waiting for startup",
                         "start": "Preparing KataGo and opening the connection"}[step]
                self.status.value = f"Step {index + 1}/{len(steps)}: {stage}…"
                self.record_activity(stage)
                self.run_action(step, settings)
                if self.progress.max == len(steps):
                    self.progress.value = index + 1
            if self.kind == "colab_analysis":
                url = globals().get("final_wss_url", "")
                status_url = globals().get("final_status_url", "")
            elif self.kind == "modal_contribution":
                url = globals().get("websocket_url", "")
                status_url = ""
            else:
                base = globals().get("base", "")
                url = base.replace("https://", "wss://").replace("http://", "ws://") + (
                    "/contribute" if self.contribution else "/katago") if base else ""
                status_url = base + ("/" if self.contribution else "/status") if base else ""
            if not url:
                raise RuntimeError("No connection URL found. Check the deployment logs below.")
            self.endpoint.value = url
            if status_url:
                self.connection.value = '<a target="_blank" rel="noopener" href="' + html.escape(
                    status_url, quote=True) + '">Open server status</a>'
            if self.kind == "modal_analysis" and not globals().get("ready"):
                raise RuntimeError("Deployment finished, but KataGo readiness was not confirmed. "
                                   "Check recent logs or open server status before connecting.")
            print("Connection address ready. Keep it private.")
        finally:
            self.clear_controller_password()

    def clear_controller_password(self):
        if self.kind == "modal_contribution":
            self.fields["CONTRIBUTOR_LOGIN_PASSWORD"].value = ""
            # The existing deployment cell keeps its subprocess environment in globals.
            for key in ("CONTRIBUTOR_LOGIN_PASSWORD", "password"):
                globals().pop(key, None)
            environment = globals().get("env", {})
            environment.pop("SWHUB_CONTRIBUTOR_PASSWORD", None)
            environment.pop("SWHUB_CONTRIBUTOR_USERNAME", None)

    def logs(self):
        if self.modal:
            if self.target is None or self.target != globals().get("selected_modal_target"):
                self.invalidate_target()
                raise RuntimeError("Select workspace again before viewing logs.")
            app = "Katago-Contribute" if self.contribution else "KataGo-Analyse"
            # Recent logs return promptly, the deploy action already streams startup.
            subprocess.run(modal_command("app", "logs", app), check=True, timeout=30)
        else:
            paths = (["/content/colab_katago_contribute.log", "/content/colab_cloudflared.log"]
                     if self.contribution else ["/content/katago_ws_server.log", "/content/cloudflared.log"])
            for filename in paths:
                path = Path(filename)
                if path.exists():
                    print(path.name + "\n" + path.read_text(errors="replace")[-12000:])

    def stop(self):
        if self.modal:
            if self.target is None or self.target != globals().get("selected_modal_target"):
                self.invalidate_target()
                raise RuntimeError("Select workspace again before stopping.")
        self.run_action("stop")
        self.endpoint.value = ""
        self.connection.value = ""


def show_swhub_panel(kind):
    previous = globals().get("swhub_panel")
    if previous is not None:
        previous.activity_stop.set()
        for button in previous.buttons.values():
            button.disabled = True
        previous.view.close()
    panel = SWHubKataGoPanel(kind)
    globals()["swhub_panel"] = panel
    display(panel.view)
    return panel


# Load the bundled actions without executing setup or starting a server.
import base64 as _swhub_base64
import zlib as _swhub_zlib
_swhub_action_bundle = 'eNrtfft32zay8L/CdW9PpFSiJVmSJd9Pu3VtpfGpX9dS0t21vDwUCdlcU6RKUnYcr//3OzN4EHzJctqe+8uX3SYiAQyAwWDeAJ93/PDWC3YOjJ3vjB8TL/GZccVuvThhkUFFRi1aB7ERBv6TsYjCpZHcMWNlB8yvzwJvuQqjxAjjhhGxhhGv56sodFgcz4JZ8J1xGt4ChFsDoNi3NvztBQ+277l2wmKA48VGECZsHob372JjFbEHL1zHhstWfvi0ZEFiJHZ0yxJzFsTMZ07CXGsZurZv8ffGyDgPA4Z9seABnsLYhB9eFAamE66eanUquJ7tnF9YRxenF1eznRuoNttpz3Zk0XR8dSZfu+vlHEtmAZ/6SJuReRmuWFCbBQb8gXY0kNlOA5ol4T0L+M+APQKwBq8VJ264TkY6jJPLcVrIokgvnEyPLz5NRXHCviSjabRm4hkGO4L/4AkmtQhhcbyAIWJppCbv6oDXXUVekNSwQgPauSOY0k6dFy3txLmDeUXMjJkdOXe1aLZzlySr+GB3l09pBrhb7tKcmgs/fNy9Pmz+025+bTWHVvPmB5wnghYAvQWHKbpOu4dOZ8HF5fjcmH48mRinJ+e/GOcXvx7A29mO8QNvZd5G4XpVa9WRlhZiLo+2l9Tqxl9GRktAjWwvBrpcB4m3ZOMoCiMAf4ajFTS6sD2fuaZx5HvOPZIdvJszQBMzOOUgFdrGYxjdxyvbYSYhBGayo16VbwFVvO02yG6Bs4vjw1Pr8uriw8npmEgM5m5ANys7spfGc/K0YgeznTgBlN3Odl5ki/H555Ori/Oz8fmUWi1h87zWEjfckQ+LChNVe0nMPlTosJOELVcCH663WLAIN5qGmVf3mssWhnPHnHtRwYmYCzA8249r9QNFFtpmhPa4YDS16cUv43Pr5BhWwAAy3lhrMj66Gk+hpkZdZbSgCvFPviPspQjUCB9g+3kuMKJ0jWHlFkBIsYkLlQN6xZbQBJY9hO7FkIlHPdiRZ8+hFXa0hkLbcGAZAiPiY2wgqcATg2dOmqYOvY4o5UgV6AyXSztwa+8B72vsIZbTr0S6wnn50gGfxbV7BYk78GhMCICGknjtIDEv1j4QvqAhzqCBhBrGg8cekZZgG9L84yRcreCFqTiOwGkjg7NR+UjFbmfJOgpyPLbZFIDwUcFMccTrQB9YrnV1w9G7AXUaB4YVqmW71Tqd7fjAE5C784Ug7gztRRWYU2a3m7g1VzUici9YrZG2f5Vo3ZWNAnvJUn6CHRj2HOjsALqrSxAkqjTcFXhEWhFoAESqRBD2jY9a8wxH/Wz7a7X84wAZ3jxM7nROmR0p0CX8p8PLUDeJdIf5Pl9+5EhXzHabxDUJaQbsOm/hwWbRVoHA6lOUvCp40pQBE8GNv6yAxXuJsfBtoDhoHBr3jK2Q78bstzW2FhsImTUfkdwGC0KyDfzwQe11GJYd3MKWLxBCmaQvp8LZjr1a6SRSSYwNbSNL6V6v5ri1ss1DJIcSdkEidiLaFpfswHgWv16M/xhjjQaMZw3eC62VFNrHhO8G39C4LrijU5R6QVZvMx493yeuR6+VuNEEbOQlzAIEVQhYLDageKOA/f57qkcLxnF0byf2bWjFLAKKMldPSgjb8VPgeKF6/nccBurhzo7vfG+uSWz1M2LqZ3y3Tjw/fdREuniFbFM9rCMfYJoRkh+svnq99lz18NVbEdkENDUn9AWeAKO8goutVXGA6h8ANfRJidewEPYti0TdlZ3caRUv4VGUgIZAujcvOPYc2KvT9conGS7ectKmjTqq+oOFnwDPsLgJ6g3xK7Wx+IP3BShy2t09+nR8KAWhaUwzpAMEhwwgDGyULbA0UGQcXX7avQBF++gUB4fqCOd2P19+Ik1o2uUK+nfGGRoUWMdHdQcGhzwziewgBuaxZJEVLwGw0NC110vmeutl8b2Puw5fA4MLfVDOcR0eQn/Nhyg6aSBPne04a9gXAATHd3hybsEgx8j6x5Mx19jKeqS5QLVPV6dUR+rdt15yt56T2u17t3fJA9BGGO3+AiT+c7gbwZ6yYxbvuuEj2CW2u/vQNtv7Zmt33m45vXbnbhDMk71k4UdxcwHU3ow9f92M4uDOnIOycfs17RkF0/nhGVdF39588vGw0+tTY6fVdeddu9fa2++03WF3MJ/vd1p7ne5iz5m7ttNvDQaOM2TzTnevv9dZtDot1p33B93+ojcYDntyFT+ugZ6NySkqKeFKoBr5jhPGSWwAvUc28CA7StarFAOA/Ag2Ae1DUxAoDCpcLEA4AHe4Q6hW7FsPLVCbgfFPfv24nhsLZoNqAcInYECeVGlyesl55C4s2H0zTp58FDy2/xR7KBQ+fjo7LFne8p5mAWD3p9OxpbWC6sjos5D+CAroEQUMnL1BFxawSQN5aGlrpneYW/gtW2nr3d/b3+/2Wbe7aLEFa7Vst93p9mD9bXs4ny9a++0+vN/rd53u3rDfA6k/3J+7/cFezx107blY718Op4c/X1hX49Px4WRsHR0efRxbv4z/QV08tK32wOpgTdjtFrIOSzT4I/A1MDu7XGo0xWPTWbt2u2MO8EcQDM2B2WqCbbv+0vzS75rAs8vGoiGl48zbw/2FM5w7nVavtegM2+2OPZ/35oM9x53bbn9vuNd35t1+q+f2WXs43N/v2Mx1hoNW32nRVCfjq8/jK+vy4gopa9BqtRSafp1Yl4fTj9SVGDpHIw7p7PDv1ueTycl0AuU9bIVvJ+PDq6OP1vTj1fjwGEu6/P3h+eHpPyYnk2LJ+blYhsuLX8dX8L7Txg3FUWkITBlLQCjtnMc7BoI5MpJH4NXI3M/PDS6EubpgAyP3F8BDvjIX4czJ4QDiJuLWk4d7GlQwFA7EvFEqMJC+8doHvRXAeLjvSc9gBrfxPwMTpn1++Gl6YU0/nY8tPtfp9OT854naYfgSSuFdipmOxIwouLyAgpOLcyzbUxhAZP50OAXMTU7+OVa2rigVS5SiLi0lwjgefz45wlYtXJzJp5/ABDgaTybWT58+fICGZz/REiF3+PkEBn968bN1enJ2ggvepgV/VQBfkvB5XfQWhBEO95lrnaXS8UCW8hqgy+C7P0Y2AY+56wOXIdmiWExD7w85L1oWvNMtG8V3NuxA3qQ1tzv7zGFurz1ozffYntuaD4aDeWcAwq3bhv04HAz3gSHtD4Zum837jtty99rzzrzjtDudoQL90ijDkhTdfyaatpPBr6DtG4DoaPz94nwjGoVu9RoWEdu2yTkdNPcC9CSE0e0uqK2ANebuBiwhO4fzjpi4Ypv/3UwWe815u+3s9wfNuA0bv3PWBA7c6p5tg8Rvg6HjsGv399odNmgtbBgr25/vu25n0B8MbXi5b7fZAgRBt+W0W23AYLvfGbZcG97vz7tO3x2yEhxyW2dhx4kFQvvPRqCmF8TD4bDfb3W77X7T7e61+73hfrfT3x6R3wpLR+jQ3bf79oK5g8WiN9/vDobdfdjs7V5/0QI9xHH3+q1+y+50B/D/ISzWYDGf91nLmdu99qLtliDUx9ADILPz5yOzM8BNSQiA9e712kPYOoOm2xvu9fYGrWG39QZsfiuwzB7vzR2729tz9+etAbMH7qLfA4IEFaXTc7suKGtttr8PeluPtV17b7EPwN39/Xa7C6S8N3dK0BknUYjeE8Bot2V9vfO+en82XqmTJnSHe5QjpN3Zx22KuHjDVv8mQDo+273FfG9uLwY90PX6vf0B6zvtIfxi/U6r7dhD0PGYAzV6Cwa6X2tv6LZ6HWCvvX02b7FBBp/kwC8aGxnxXW50/Bmi6RXbYqMs2q6Vjsjfb1vkEcn96fZq5T9ZREMWt97TAMWtH85BtVTmWMPIGkrymWv7mTZFG69h5Iy77IsUZtG4wrHywBzFGwv6m+mHjyyqab5gFXWDBn8Zab4Izb8vy9H5C/8UtcKDbHij4A5ezHaKfo0ldGTMmQHarxEujOc4jBLm1q7fF+E3jHd8WO9u6i8qFoB/dAO42O4ah31zLcj4Jt9MM2M3tE0psgBAGW8bmkvSvJFrQ4QMbYrbs3J1RBMKKnsxhofczAKV+gk+2H7MRLjXV0A2rDGvIBa5hHlsscol/o3KZS7pIV1n+CVnWljzDV4RWaXoHSnp7ZomXEYeFb6OjTBKyaTU/bERjEYufOVidrDV1PF/pSxK85RwVJS4Q1QNNchyPwV33Vmnhz9RzwsZN0d/6rNyrb5wWx59IcYPhrM+Bqt+aA64t+Hw8tJSviMuNpqH5CFjWAF99yPuSTYPV6uarA7TeCDDXZUKOx690xYivsZFMXo1GhgPAt3M8hbW0otjUAlkfG0WzEFAWd4SHX4S1Ak+cUgRhRKip5oUk8GDBwrGLjp3DnBGZpv7d5rCD91cz+HHutMxW10lOmzXtVZPyV0YjGY7e2a7TSV1kwUAehVicOT6ps5d53wgor90cPzZtFeJ5QVxAuZ1TZd4jxjgzArBdUA+psy71T68a2K8NVfge3Mo6RbfhstV02UPxYJbKGnnXjt202FR4oEmgapwvpT2lXglNrC58lbphHbQGLFX3jX8Hbh25N6MRi2z3QWlgbvU1w+eE0YBvu11zRZ/+cjmcejcA3WPRmD3dok7EDbJ9wTYxMAFQN91bOeOUbEg4+MTdErxartIvyLy81zlS3yxbldri7v2rIHFXXupD5vD04DvcjsL9i1WEgU/nZwXa6UuOFEw/vv06vBoOj4urx//trbju0XcjMIQlp7ewQ65WgcI4+ji/MPJz9LDl2uqPNDO4jYdu6ibTmQ3JxdzKmSxehmfTBUmDHs6S1cPcMPvR3eEDq96JruIBx//y3iG+i+4yAt/Hd/JPcvD+OTSy+RQqQ7iO+b7ooM01akqY6o6WYpGB//V9bHxnkVOVMnIPJG/Qg5LUZknHThAB3riUUWuBEz+iEdERebRgcDDLHjOdJ6KQZHUkClNES/iqxa8xEwBvgY5vP9f4DM76hRHjaqJcGFooWitYWhSDt710EqEwYsIrMkrSo3pEWwUg/LrsBFyjGiOCUJ2bMThOnJ0iYrJb87dOsAAouEloH759nIOvF5UhWHabq3d6nSN9wb+U28Yc0qBy6lDfEzmeoX5iDUCmZ22qHDHvvBfNS1Xh/IYnixpNYlxsy8rHsjn09OyoSgrA+qY7As6uHleCL0ARpqQBmmh09z4f9tQ37HoFnMQ3LXDeFbGcpU8GTLyD6BT6oMB5IamdWE7yRo0glFx8TLKJq8mNV7cJTmQsqhU68zPALSTJuotIOx5TiIuK43aRNXg5UBBN55z/bw0jFvA5jMf0ItINeHLotYDBFnDKF2UkU4L1OHKxhw4c3nvelGNP8RiA9FqWeF9jn1kVxL5QNlS/jWzkqBH5DCzHQ1l2yjue+gjnT/xEfLYi8itcdPlL2F+almIxtN37IvDVokxpn8wYdDGSI1zUNH9Ecppl6hN8EDRPagV0LqhVoIyDTH5F7NOvjibB0VYXAe+F9zXhBqoIV9kxi5R36SayDaseL1YeF9qfAHoN6iwsx0T6in6h9/VYDPzOtaG/QxE9GI0/1qFTmEdwjJg9GuU9cNgykTz8BZoiTs5zsKvnu/buz2zZdT+3m7/t3GKkUbjy6Bv9bt1AxQDn/3K5r94yW5vb9/c64sYXJMU9t026VJ6D4cOLhWH/n73feoFkUyM8lFgXNkEFfOK/8u3iBj9SPwr8ZWhVeLOOSDwSPxaPDcoFI+ypt1pEd8GAbEKgxj2ENWDFSBFULL1NFtZ/uFZN5S4jVQVzv9d02CsEzG01+lUreUl6fOKElNZDfX/ElVTosyFmkaY5Gig4g5tfX9uO/emaVa2e5XKaFlUapkSaO+oh2YziTwWj/bwl0BmvwUPa6QkGylplCWjIn0YzQtA8jMMBGaHv5CCZzvvGtkeM/lnmUTU4uLnmRQtZDWPCkEpYSvfdhivqcmR/ModvBVzKEhSVm9JHdlywmDh3Vqo99R4LNpK7pA7xqRANYyl/cWiQDTxZfHSCxx/7TJrzgLnbmlH96oR+WGkhBAAMeUd9/j1jZZsq3dl8NSmfKat3hxMQ9gKLpJmsF6en0+obMrbX7LojJKZgI9kIadiHKw8z2Ew37XIoc0MAARouzhmEFMJdqopUGjnufYxAZuGwKj4EJ49wOuXFxxALrT9oicsLyihFWqi/hVh8mRNH1c9YzySI1JHfkkycq7CCF/U2g2jJJ/hvVGWzFAvmbWG6uDM/vITwp9w8M/ZDlP04v6GRoKeeLo+nlYw/w32f00HLxvkSYd7WGBxJ3S4YqrePm89mRd+QAIQV0mg5GOiswTCdyE3QjoKOmjwOowsVeTHDSR5KEDzN+UT4SPOQ8q2VDgon26GZoT2vaAJ4vmkYy/C5qmJ/LKL6anICGApP3uxlyj4aVoMBQKeCxN/ec6jiyoGAWk0SCKXqMZeLKaPoQSazZF5QbAZUnmhOexwd5ngTiJjxiVvhEydrAnnBfvCHGUcUBo0aO64av+R1Had4rPK0dEwlP8uG60QJx70yIPGkJKoVlzDekmNwjYTdW7EOt2zp6JBJ2djMjIUa/W6bj9dH3S6goGSo8dC8VD0fiTrgE61cIcPMivrGXp7MSmZd0exlop8IEofVOCVnr5RF3e4PjuifGETJV1c02AgoXABU8+prOhMoLbi/EoMv5c2ajmjkdHJ9aJrF59QzMl+RboVemX55A/IJ6oGsEFn1rZMdiAZ6QADamTLs1ww469/iy0w2zm5DcKIH/Oj033aLHiPNBdoX+6KQaFZvpC6GcwnyCW3PIiEBZoXzeSp5rRK5cpBtcinIYkxCWcMCvvsptA3LyqzCkxOMW+SOxEnja1SX1w9W4uPTFbTJpKrp1awUbF5q8RiFswD8UkdSiZXLlcbOPgqhAaYo17SRqXRKaag21Gznc/8mCcZf4BKQQ2U/icT/7hZkc0zrNKuq3xf6ZDFom3j/dIXtcoNJhXwYauV9YZv7WH8Fr+iDK/gcgokKRoT1ktDy4sU8Qypea4p+paevMqph8CNcFxgNx4f09odnhpX46OLs7Px+fEhvjiAGiqd01BkR3qePmGuO7Sl0oQspHiuVJvoxi5tgzIR8egRwUBmZL7/W0Frq81m7g/1zE4rrIJ9G49gBMcX08PT0+y6FZRMXEZt7OIkarteRyTpk6LJcpaTlVtv9RtpLTVORfLGhd0W1zL+AylHYFGQ1+S5+UFueTO7N8/aD3LTV+kSpMiDXdmpZ3WvHOxc81TLYQHqOPIwDp2dVHI2lelbY0hFKrZqkZEgadwm65fTyjLRmjKt4Ku3qlBI+ARlRrhyIEubOA3LNhSQhpGJw9Zz7hRxIMj8p7f6gP5W2YycI18XOVH7dWHSSQiHgnDp4OrFwb/mPlODLqS+oOjJKpIpNyuGsQ9K8LAxC4aDL8bX698kyuuqGSVHL4hhMalO8SjbAbrH51zwMHxHUWNj4UVxQkdAYuPDp8kY6USCEv70GP3SPuNuwCWZ2ww5NCWmc+aLp7FssAUjPCVC6nCqHaQElKG5MoJ9lSg1wgZBELq1Vrjf6xW8VYt3FI6b7TynDdAHhIn5IMPfVSiQPJiDPH2UD0IBxCpghd7LpaBUj1H4jbLCrxJhCk0luRS6giEEpevxLY6nT4FjMCfxn0xDOO7Uiou9g5rsNn486aYrwUCzaa9WFO1vCqDvitHSzP5Ui56ZbjmBZLN73kIl1YGWMiRIrPGcC2VZZ3rJZvDgQV48dMWEZRXDP3j3BHS4YhHxmjAQJ26ZMV8HLkYEcK7rBLUSIwlDMwVXK50XZ/fEcNdxJALmc7r5QA+/1ys2QinM8qqc4PNrnJ+/tnkyJ7619cnco1H728G//nP9r1ZzeFNvz2ag0cJfHXjL3/3nv1B9IVf2qwG+sQx+CTrn6jEFvvAGjWcVWxZiG3VhkfjSjJceGY/5ISuPhDiv00jfzIX6toXXokK3LD8qA1iqKEidpVzByg+uQmsrP6+TdpMvqOyGoH6D3NmsGeXUKA1vXGdarAPagRY/6hjr2b08hwh0NfqhUqx5DlXM83xVfsyBwd9rxy24xYJF/RZYhPRXVxU7qzXpkmYrPVzAlmH0hG+7rWFfvb7lNVPfkqoPU02lHtZpp4ngjg0bHvQA6xFUyvCRDyNNx/0R2KYpZ197/z6PCMDbjzyzC3pw1hEyghp2SFcnxKNOK61hx7ceHijHcDwqobehYoV08FlkKMmTzx/g8fDypGH8yuYTSkLSfh57MXQYwF4rAjBl+Ekd0r70Ye5TIIMrUSL3n1hotCLV8TBOBxhVsxI7vteDB98ZJ3jZAh51PTmODWB7BhjIv62JmaKNKxgqbGLfQ5aItebMD1GVCSkxU4yajtpzw5QFqK0c0EnvazDXxWnva23a4my8eQrPZNLf4L1HzyJgyPuyYBgYhxKAtNYp4HxL0belwkJI/bE+FGAlZQ1A5YAhbtNCYJjvTx9GBGX6dGQmCd1bMOLH6ZGAfBaM8sfuJA8DZZpZ3Ii0POJi0GmtrDSM8Dy4BFvPDEkcTgbrJ/Bi7kOUAxs/IBkrLo3pASwfJ0YzX+w3kaaZzZNH6IAfm3Y2nXfGv2pZd40+FMo05rnEZXUoa6C8hsrSszgpxFYSJjYdJ2i9UpFfqlGsKYgJenUYVHCLNTiGY+D7PFKeLeVad8kI7CiguwgKJT4elqJmWEYOw0IxjuoJlry6hqKK6ipzPPaCdwJVV5Eut6py4JURkER1BWBOYF6jIVJaR24jgeUiMnQiji03ClersjXg50l4FjILZLZ62UEHlWEgA5KoBj1anCSL/lqiV9jOC6ow2/n+48H3ZwffTzTVhfYKAWJLL6kx3DIW3m6F6kkcoyjkqmsmGU4mNanaPNdernsuswD33XVKTDfGDyPpvsrX0ckH+Q/6PcU46hu7FhRZ2XlKsfnuZY3CamL/PubE8QK9/4QtiZHscPLS+YLQBfB1ujT0FsaKb3UUoxZAkyOPkjbXBqceTpwucOUaPdVfskjAhclNGMcmk+qoXBs3cmcZmsV69ZKsjetnLLl+hwN/d/NyYzynA34BJUiMsCodBymJLHxLJ/6a3PL1oo0lizBspn6DnZ+XDwdlCVT6oY0yoWLarpt2XlGPxIvES0nlxztMs0JSKDarG3/F4/9FqssOwwVFx47cEgDmKgQZvMhGtsTk5EGFFLEo4yydUp9KUauuMBJVsRNVkUdwstuJN8jNQsmZhmFJXQgXZ2To94XJPxI8CvKsLkMRLw3Y80sxiKdaU2XVF4XxZFlJNC9thjNUzQpTLFeRNqOlQksqb5Tnpgma5QEeqShfLUzhtWPUMWGtQa210AL1UYuxfZ4B0zCwOytYL+eYIVbCftVuEUeQpA66OdNQmc75bnPtNhPbpmCumjvnkVyKA2tR75vPEs5L81kxTyuIa/UXfmKBe0p4cp1qJi4Nko8nBFdBKpCUjr6K9KDCiK8x+h0F57wVlwE6nGxTG+/KNITRGgo/fE0Bq5fVJjHLNbMn4qGznals4KY3Rwq08+kZnjtKUQZMmNMO4epZwBLKkqr15nhyZnhCBNPwjsK1z2ksXdjyYcnEvpL9kEfSyn5Cn3WOonVDTmQqoVjQXmO4CRPP47Kl3BTTEyFGSQjROuD6Qr1qAHrssJp6yv1HuT7ZF4+uiMPIA8F7Lu8n6/TjyKNGBfOr3JebIoh7Ump6eEtinNJzKe2nLrNEZjvrZNEcwJtSki1Cd/HceK1kmckEkmusL24QBjBusLg1YJt4SIYa46cYdBK+/S8jtrIp30EgmcJeaWCAO2bIv1zf5PDmHUhTEQiT+5RqJTG1HJziYNMBF+GJ8aB330tykLbbmAV0KF2Xtic/1mfwDgxRVrYZFayIbPTrspwK6XQTlyhWZlJsTqC4KfDijQEsfWAyXxIYMT/Bnh1DPqJVv8mjtBiz2IS/2Y66AY0H/OUh2kbufjJD6DL8xkVMBjDLUFtFtaf2OnDuZIwHpRptQ0Pmm+HU6/XKTV2gV3FcM02doCWsFSeOV7TGjeJ72scjCa46R0OrjwkZb2qAyR3bN/A9QNuo9PomfpBI/JNrWi+39/IumZvCkee8gzClvCL1ZLGO1WuUksaxAqp7422NADPFRjf1LYkpJ13UHP84kS96EL61YuZ+OaXmXXGm8OflhUQGdQd5OwtXKTdOumx9VC6NAAilB2KdWtGuQLGNRa/xhBL8imwbB8NsbmHOlHwDHd/ndd8vaHdhl6bLstK1IaLYI9CJMWmPZS4ZLhk4AisZOEYBvCBPy+ViyYW5ZNMqEWi92pJCNyu0EVmKXvnEU+sm3yCjnZe09BaZxiUjLiwNt5ZzujLRtHMfhI9AmbegXKEuqkN+Ke2+Gn15m0qqn5jFUtMmKLYJHSDUXiuZki8Iwivu/0PtqmK6LtpifhaRW/WjaJXrkAzVdJGXFefvUFebiT3QMQfVBU779Y71ecgMeDWYV1eROqUF/BTETrhKQ6vPVITWA8dC9bqBePYeWEQufi1SkP+DByQ0Z0OMeeI+RVws+mCDFyfSpWc+4GUcYHVWLYwgCs9N3Rd1BKKGsqHdZlIrTIo8VZmO6tWtyvd6Qfpw40Fh4JUmmkUrx2BS23yiHFLFptFtr9ZuqeKmFigsLhS5xjpHRUS01QLq9fUoeMyQ6ku8HXkOXeY3y+siFWGAMj+4Bn5LL+pmqYatBRKPeb/cNylTejMWvIp32uJwq2MHeJ08I9+sOFfm8jRi4b94M57Ld+ZrvsU3UvR2VIxedSTlBmIh9TiWUbZcyWy0rGoBM8EMLZSFOqjy1RRaYSzL8oJFGGcFAb4/wde4jtc35bSYNn6FFvOxsxu600U2vm7daL1WLm8GmIyyVUGS5dvAUhG5KmCqwjbQ9PBdFUCqc0pV6q+oiBk9ZIL7pHwTIXZHz3wk77L4fnfzgh/tQIRka/B3UFylNJUI6IqV3tZvm1rMWfAZQZ/VAbz4eI2uF35MDcuBE5CzumIs6hKBUZUHXmezDS7W65UskT6mIdpvYIFvxcAGv4ESD6kPOKfvKm9xJU0W3dGveKMzCZcVQQjAFJhSti4VJPq4JhEkNa3H+uaxSaSaXkxf8lAaUqHfsvWqvzKHt6zHN7rEdOOVvLBSU9AchTyxdrObOmOg/17TFIB8o2kqSnPODJE7crON0aQQopuz8FKYs/IbLwld/o2yHg+5bCDiV0z8LSzjZXz7JsM4KrWM6ZIT5OXxrbzwpIBYlbbTEHNLQpgvfpTsDqw5H0lC3JsgjtcQoINqQZ7Df6kr6RuQlFmw9Co2zaij/goeTpqi0pA3TaBKpwbslUNVJqGAyY1M7v95Q1eK+io7UmdMsz05tu/HuzFzCiUuGuOFVq8NRDYoH0kpy8+muB+ItqXJDj+Wfhgmy1Fguixe2UENcyZ1liKpBB2DWtKa7i7MRlPq3xQvmQAN05foIrwkgL7zEdAmlGzREx5E8aVISiC3nXs8kBW4xSjKk8d8TXulyLGfH813xgS/W7SiWI2tfwAPBkmp69wfTh+zukUWai/wc0R4CQmmsZoFp65ClcmNklqps1Qi8dbGryTU9HYN4SOxmBQtcdnBg3LLOheaRJ9QdaRwg1TfbLfrIJUrq8rNIQTlJXcCn4bh/XpFkm9DBys7jr9lYFnk4oMFi1nTx8u/Gpme3OxtHraENeXVXxu43tO95/u1+muDLY6tkN1QQrfSf0R7ks5dpvGBqiWtpslKunyvw92SMDfLo1wa1DbRkIomW/nywbS2+MWjIr27JtncSP5ImaSoLBTmXcVRUh55F5bnLz7PdsJ7zGrgh3sy2XJKSvKwrzg1q8L6mWsvsX72oy4yv24dc1i78oG+Ihfeytf850vVXGSrekkcHApqOX7/1jzDKjSozED6Qff1MQfdF3Lk5OTEh/p1s9c6uKmcAG/QUNdFWQ6Yo/GokGlfmCC0p95q9CVUjM2SS1l88uu39E3u86HhY0xaoBxfNgsN0znpwDr++IsCmM9LswNuUz4n6jOPtGtJfkFTM175JAgbZKMCYFkvlyUkhnNNCZ10sw7+wI2PBej5xWxKYelR/ia5f/kAbrLJWLlB/mZJVfW3ckX11b4lBH7PohyG2gINjlwJ/KZANpnrc0ozO8UzTAyeD+Sz6IGcEYWBldhMpFQz6zFOfdcH6REPffXzfjmbrlDTEaM8exXnC/KJhs/agmaTDq9VN7rbLS7ux8q0+6pE4er0+3yLjFLGKwsbVZwfwZsjqa3yBlUBz7uGqjTAKvZdJgPfbGXytDNKyy5L19YDUuJSQrK8HGDv0pipkpApUZCBWkNlatRutdsyUXKE6R/8Z/36oN1p3VRexJLjFlWmOz+UR7SuLPh0GOLIhDjuXBEvzUZWi4nq+cCi2MMyu7HOc9e1BMdKJQN94tw/ns2U1cFmY6/VbjQF7VWVryqLVUFAJiSCBsKkxkVPY7RcthQSXMVNuXp+a2Yiegpm1UTeGOHaKiZAg6h/c9z4leW1yoRZajIdCg72LqavvjahNv8YAJ0VJg0ThGNI339Lz5+941/QNquDoSnxyPBnSkLbBED/Pym8gRT0SJY+D0qk4J/ojmr4bVoT/+qWertlEnDZfk/jVKVkqDXd6CWsWk2tPVcrRcxv/mQEDJURzvYI1egpiPGz1BVJIlpGyRaTptrXHFGbA2UKIdcKGZsbiCleyxpYu/ZKILJeCSYXwchAfZZud6GQohoGmy7DpSnF8iubYmNBE9c3ecV0g+s/02GFZ1HK5+L5v61ilmlC+WbEVma0X4n+yvLEGxSHi0fPaPOkiHlHb99RiLP+Qie7P4tAmV5Jvn1Xf9kQsctnm/PNXEiPKzmHvMmtlmpw9FkZpcP9kZl3akibUhrKnRfb6Kf8ds9WY6vKTaOdw7DLb6G8/g12NRL4b8gmpHkgMyfwGe//EebTdesGvWKpOp5zxHAwPKvGLuNa5eGj0i1a7cYqp4uq0yl/yMGU7CGMKvqoGG/RVZc7U4WnkDQelju79Ir14TM7WK/eanvkXAiqeoVXJwWHH+DY0jgsMSY39Zz16mwYgljw3z8IYVyLocwCwOiOC2IyfII13/nO+DHxErA3ruizOCAweZlRi9Z0QYv/xC85QOG5sgPm19XX4MO4gfxyTZ93aGh37TXEZXBA/dzB2ih89L76Y/TpB4Ssi8/jq6uT47G8oBZ0TvnN74YBJPHAjLlvB/dotdF9szhIdPPhJwH4sPkXMaWRST43c/U0CybTw6vpp0s65D89ORtffJpak/HRxTl9sHdIX1Uejy+tyfRqfHiGVxgefpiOryy8juQf0hmJAwKBRrYaco+ncE2uFn6rAAzqnrGVdPGJiMGnq9OYh2xmwfTq5OefAShOmIM/Hl+eXvxD++QTxZ74JEQuPuBBHJ/j35dLnXalp4mOfM+5NybMB85Pine8AgVJRjf4YiMTFBuGBQ/Qexib8MOLQLnE+9VRYsMzcN/zC+vo4vTiirPm2U4bDWReBMM/k69BI53z2225bxEWZynHjx8gSb8NQ0cF9Fx7uv1d85Es3ewFjN94Q6PmiFAXPmoyMHgYwX/ZKwDzV3gj46foOYqONJm7cJM81mkASJd/OUI7zazfNI0P2dtfMmEJqisRmETe7S2L6BIT+n4U7CaJwu+kPxkdKDG/1ExeuoL3K/Hby2D3G+rqfSSfJYsYbG283IlIQJhjABhmLPuQIW3OLukAiPIgFz/uM+WjRFKXV575Lh8XquO4sPygD79kvp5i1U4S/BhJekF4u2F0N9+/u+kbA/ShAhnX2ct+XqAyfrpQkUdYHfqAl2jjyoNwH6fTSzzJyAGZHBHlSt3v/FiGGIhYdoWeZ/HjBZCKh3pkFLQ6jZTfp+ADF6r16uqbJ6CaOwkSE/+qWuzAGidkT0qcy7KSezpB7/uXZNCzWfx+hH9dv5vNYO/Xrv/Ff/xQF2/oGFQKHwWFefbpdHpyenI+Tq/GyvZXjKtky9Xdm+LSJ+KOWKV0tFQ8m+G35mazWvVIZ/XcYLUr6bM9FMeXq5AboHJvpx8Ek4wRw+UgLCweX6vJeW7PHzOiAY211UqPCTUU6ugEXHMR+j7etlT/v2Kqv3H7bM3M/8G/a5l7GAQW6rmvN21mugTVxBsINZaavswcd1dqicmvmRfOoBHvuGG4NluGAY9mmrQ6tSKPBqUH4QlKEReAou5iOo8umBG7SmIX1I4dEujkw05b5+/n42Icb/k8D5MPmNQgZLl+Y/6zBuDFcEMWi2RDAGUaXOgfc00uCUVeA1P6EQkFisdot7qAHhnfUeI659nE9L7/nkwQOlFYPilTu45fYUWcKtJnqV2JnvlGZEHbUyEyzOXcwKuwO5n7ciz1GCF6UOkWJ2qz96fz2fv8jnFteIWKCo5QJpEOYUtRO+2TlSIUzu/F5KJUffsjp/fktyqvLQ+K6mNBdxNdCa1g65dBl5/gJuhSeRfBDFpBPMstDnFrAKWBxAcFElNqOvqG0+dzkIprVBIo18AES9fFG2aj9PPR1/8CgXDzw9+A40rOG63Jl0rb80APfzH6TG5G0TAb9Zv/fpc/5c6rCrU3HXKOC6QFUsmiduqboPJroFTlutm+yek3itMTOrSPZog9izAOsveWg9ojdpkMXDWM+ToxTgDj8giIvU7CJidk7TJusAPSXCUJDrNiDHsePnCCsw2JVSBmfpkdXlxKV+UCpAA1PdiECSYpwS5/jEXVXXH/sFhjbbDw/9OTz2N+W58hjCADjKBJZq/8CloobgIcRPZopWlM8TSFF2tTAfXLe7B9dKSHC3p/FPr23MCDQrhpjESdPkZZx3ko/iJBwKNzWTmobTDCfqWhhLF3bVmq+LumQDfo1PaI9OhGvYrl48bGREyVdSJ2lHrmQ6br1Ef6dUqzINVVSwN5GM5KqxtNQ4P0V2ODWVqlLMKSTsSxV8Q195YDNJduLeVG5/MGuC9xMWculw1bVL9FFrFaRHJ2qk96Fb2LvNIYvziY/5BGITLhLQQDKs1M04iZUx9OmnNa1NyZ++psXrPSRBYE5RwWEiHIHufbS0aIc3mff0Aqr6S9YtKuvOa30jVx8Gpqsz6D7Hnp6gEp6v9DRyRo4xf2NA/tyKU7LqP1KinyWEzOxKNgiqPwBVd+i6x/+S1Dkgwpm7+Y55mTMbLJo4uzy9PxdMxPoIrPKQopoVWbTC8uL8fHxk/jDxdXY4N6pFHiGTl+MD07wUO1DuRCR6UDrBfmyuxW8iaRVDBBLfwC7+xECgBd0aPPJkqmT18lKJNXHxBXxuTXj+t56lJEmXKQE0fYXH2cLRXwIlUt5r/ruRpaBV7OvQbqLuaSpSVbt6L/rM8B3VB+2ZzOw1SqAgeMdZmLX51ET6BQjfihCo4ldIVi5LrcEYol27hBU7toFnyX/+LIq/ZZ8RPpeUstezPztvBx9OXw8xB3Xv4XQ/1gxA=='
SWHUB_ACTIONS.update(json.loads(_swhub_zlib.decompress(
    _swhub_base64.b64decode(_swhub_action_bundle))))
swhub_panel = show_swhub_panel('modal_analysis')


## Before you start: payment and usage

**Modal requires a valid payment method to use a GPU**, including when usage is covered by free credits. Add one in Modal's billing settings before starting KataGo. A new workspace may initially show only $1 of its advertised $30 monthly credits. Check the credits actually unlocked in your dashboard.

Modal charges GPU, CPU and memory use against the workspace's credits and budget. A budget can stop the service even when credits remain. Running analysis and contribution together can use two GPUs.

Outbound data also counts toward the workspace allowance. Starter currently includes 1 TiB per billing cycle, then charges $0.04/GiB. Cached files remain in Modal storage when you stop the app.

[GPU requirements](https://modal.com/docs/guide/gpu) · [Pricing](https://modal.com/pricing) · [Budgets](https://modal.com/docs/guide/budgets)


## Advanced implementation (optional)

These code blocks are collapsed by default. The top Open control panel block already includes everything the panel needs, so you do not need to run the blocks below. Expand them to inspect the implementation.


In [ ]:
%%swhub_action login
# @title Register login (runs only from the panel)
import os, re, subprocess

# Logging in again invalidates this notebook's previous deployment target.
selected_modal_target = None

env = os.environ.copy()
env["NO_COLOR"] = "1"
env["TERM"] = "dumb"

login = subprocess.Popen(
    ["modal", "token", "new"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
for line in login.stdout:
    print(line, end="")
    match = re.search(r"https://modal\.com/token-flow/[A-Za-z0-9_-]+", line)
    if match:
        print("\nOPEN THIS LINK NOW:\n" + match.group(0))
if login.wait() != 0:
    raise RuntimeError("Modal login failed. Click Login before selecting a workspace.")


In [ ]:
%%swhub_action workspace
# @title Register workspace (runs only from the panel)
import os, subprocess

MODAL_PROFILE = ""  # @param {type:"string"}
MODAL_ENVIRONMENT = "main"  # @param {type:"string"}

# Clear a previous selection before attempting a different workspace.
selected_modal_target = None

def check_modal_credentials():
    if os.environ.get("MODAL_TOKEN_ID") or os.environ.get("MODAL_TOKEN_SECRET"):
        raise RuntimeError(
            "MODAL_TOKEN_ID or MODAL_TOKEN_SECRET overrides workspace profiles. "
            "Remove those environment variables or use a clean runtime, then rerun Login."
        )


def modal_command(*arguments):
    check_modal_credentials()
    if selected_modal_target is None:
        raise RuntimeError("Run Select workspace successfully before deploying, viewing logs or stopping.")
    profile, environment = selected_modal_target
    return ["modal", "--profile", profile, *arguments, "--env", environment]


check_modal_credentials()
subprocess.run(["modal", "profile", "list"], check=True)
profile = MODAL_PROFILE.strip() or input("Workspace/profile name from the list above: ").strip()
environment = MODAL_ENVIRONMENT.strip()
if not profile or not environment:
    raise ValueError("Enter both a workspace/profile name and an environment, then rerun this cell.")

# Read-only check verifies credentials and environment before any deployment.
# Explicit flags also keep subsequent commands on this target if the active profile changes.
subprocess.run(
    ["modal", "--profile", profile, "app", "list", "--env", environment],
    check=True,
)
selected_modal_target = (profile, environment)
print(f"\nSelected workspace/profile: {profile} | Environment: {environment}")
print("Deploy, logs and stop commands in this notebook will use this selection.")


In [ ]:
%%swhub_action write_app
# @title Register write app (runs only from the panel)
%%writefile modal_katago_server.py
import asyncio
import json
import hashlib
import os
import re
import shutil
import subprocess
import time
import urllib.request
import uuid
import zipfile
from collections import deque
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Dict, Tuple

import modal

# =========================
# User settings
# =========================

# Fixed T4/CUDA runtime. This notebook intentionally has no CPU/OpenCL mode.
MODAL_GPU = "T4"

# Main model preset: "transformer_small", "transformer_medium", "transformer_large", an older convolutional preset, or "custom".
MAIN_MODEL_PRESET = "transformer_medium"
MODEL_URL = "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz"
MODEL_FILENAME = "b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz"
MODEL_SHA256 = "c04db4a503721d948bb720324f3cbdac6088cc9eb243632f020e4b6846f58995"

# Human SL is optional and costs extra startup/download/storage time.
# Use "official_human_sl_v0" if SWHub features need humanSLProfile/rank-style analysis.
HUMAN_MODEL_PRESET = "official_human_sl_v0"
ENABLE_HUMAN_MODEL = True
HUMAN_MODEL_URL = "https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz"
HUMAN_MODEL_FILENAME = "b18c384nbt-humanv0.bin.gz"
HUMAN_MODEL_SHA256 = "637746e44f0efe00ad1245a50aa9bbf0716efe364c43965ead97bd6835d84ab5"

KATAGO_RELEASE_CACHE_KEY = "v1_18_2"
GPU_CUDA_KATAGO_URL = "https://github.com/lightvector/KataGo/releases/download/v1.18.2/katago-v1.18.2-cuda12.8-cudnn9.8.0-linux-x64.zip"
GPU_CUDA_KATAGO_SHA256 = "2cb197fc9bc2050f29112abb5b83cdbad63936cb4605d6e199772aedc9806c02"
SERVER_PORT = 8000
KATAGO_WS_PATH = "/katago"

GPU_MAX_VISITS = 500
GPU_SEARCH_THREADS = 4
GPU_ANALYSIS_THREADS = 4
GPU_NN_CACHE_POWER = 21
# KataGo v1.18.2 measures whether two CUDA NN servers and a half-sized
# batch improve this exact T4/model. The result persists in the Modal Volume.
AUTO_TUNE_GPU_SETTINGS = True
GPU_TUNING_VISITS = 200
GPU_TUNING_POSITIONS = 3
GPU_NN_MAX_BATCH_SIZE = None
GPU_NN_SERVER_THREADS = None
GPU_CUDA_DEVICE = 0

SUBPROCESS_BUFFER_MB = 50
ENGINE_LOG_LIMIT = 1000

# =========================
# Presets
# =========================

MAIN_MODEL_PRESETS = {
    "transformer_small": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c384h6nbttflrs.bin.gz",
        "filename": "b10c384h6nbttflrs.bin.gz",
        "sha256": "0ba27eced5180b3e3d0b898b280c541112989765e789d1eb6cd0d31b2b2c1229",
    },
    "transformer_medium": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz",
        "filename": "b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz",
        "sha256": "c04db4a503721d948bb720324f3cbdac6088cc9eb243632f020e4b6846f58995",
    },
    "transformer_large": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-tf3-b11c768-s12002M-d6304M.bin.gz",
        "filename": "kata1-tf3-b11c768-s12002M-d6304M.bin.gz",
        "sha256": "4a6312e80faadee7b7dd28689a2e87a1efb4640c10132f16290da7a17b4c6d9e",
    },
    "modal_fast_b18": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b18c384nbt-s9996604416-d4316597426.bin.gz",
        "filename": "kata1-b18c384nbt-s9996604416-d4316597426.bin.gz",
        "sha256": "9d7a6afed8ff5b74894727e156f04f0cd36060a24824892008fbb6e0cba51f1d",
    },
    "latest_b28": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b28c512nbt-s13255194368-d5935380940.bin.gz",
        "filename": "kata1-b28c512nbt-s13255194368-d5935380940.bin.gz",
        "sha256": "c5bca453d7b08ea8df6546439325d4dd681e77d975e1da3f7593d771147b73bc",
    },
    "strongest_b40_zhizi": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-zhizi-b40c768nbt-s11272M-d5935M.bin.gz",
        "filename": "kata1-zhizi-b40c768nbt-s11272M-d5935M.bin.gz",
        "sha256": "15fb3baf85cdb6578e6c19b65e6201ca906cec3ba5fee19039d05221d57eb0e8",
    },
}

HUMAN_MODEL_PRESETS = {
    "official_human_sl_v0": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz",
        "filename": "b18c384nbt-humanv0.bin.gz",
        "sha256": "637746e44f0efe00ad1245a50aa9bbf0716efe364c43965ead97bd6835d84ab5",
    },
}


def apply_model_presets():
    global MODEL_URL, MODEL_FILENAME, MODEL_SHA256
    global ENABLE_HUMAN_MODEL, HUMAN_MODEL_URL, HUMAN_MODEL_FILENAME, HUMAN_MODEL_SHA256

    main = MAIN_MODEL_PRESET.lower().strip()
    if main != "custom":
        if main not in MAIN_MODEL_PRESETS:
            raise ValueError(f"MAIN_MODEL_PRESET must be one of {sorted([*MAIN_MODEL_PRESETS, 'custom'])}")
        MODEL_URL = MAIN_MODEL_PRESETS[main]["url"]
        MODEL_FILENAME = MAIN_MODEL_PRESETS[main]["filename"]
        MODEL_SHA256 = MAIN_MODEL_PRESETS[main]["sha256"]

    human = HUMAN_MODEL_PRESET.lower().strip()
    if human == "disabled":
        ENABLE_HUMAN_MODEL = False
    elif human != "custom":
        if human not in HUMAN_MODEL_PRESETS:
            raise ValueError(f"HUMAN_MODEL_PRESET must be one of {sorted([*HUMAN_MODEL_PRESETS, 'custom', 'disabled'])}")
        ENABLE_HUMAN_MODEL = True
        HUMAN_MODEL_URL = HUMAN_MODEL_PRESETS[human]["url"]
        HUMAN_MODEL_FILENAME = HUMAN_MODEL_PRESETS[human]["filename"]
        HUMAN_MODEL_SHA256 = HUMAN_MODEL_PRESETS[human]["sha256"]
    else:
        ENABLE_HUMAN_MODEL = True


apply_model_presets()

KATAGO_URL = GPU_CUDA_KATAGO_URL
KATAGO_SHA256 = GPU_CUDA_KATAGO_SHA256
MODE_LABEL = f"Modal GPU {MODAL_GPU} CUDA 12.8 + cuDNN 9.8"

APP_NAME = "KataGo-Analyse"
app = modal.App(APP_NAME)
volume = modal.Volume.from_name("katago", create_if_missing=True)

base_image = modal.Image.from_registry(
    "nvidia/cuda:12.8.1-cudnn-runtime-ubuntu22.04",
    add_python="3.11",
).entrypoint([])

image = (
    base_image
    .apt_install(
        "wget",
        "unzip",
        "p7zip-full",
        "libzip4",
        "libomp-dev",
        "libgomp1",
        "ca-certificates",
        "curl",
    )
    .pip_install("fastapi[standard]==0.141.1", "uvicorn==0.54.0", "websockets==17.4")
)

CACHE = Path("/cache")
KATAGO_DIR = CACHE / f"katago_{KATAGO_RELEASE_CACHE_KEY}_gpu_cuda12_8_cudnn9"
MODEL_DIR = KATAGO_DIR / "models"
KATAGO_BIN = KATAGO_DIR / "katago"
KATAGO_EXTRACTED_BIN = KATAGO_DIR / "squashfs-root" / "AppRun"
CONFIG_PATH = KATAGO_DIR / "analysis.cfg"
MODEL_PATH = MODEL_DIR / MODEL_FILENAME
HUMAN_MODEL_PATH = MODEL_DIR / HUMAN_MODEL_FILENAME


def run(cmd, check=True, cwd=None):
    print(f"\n$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd)
    print(result.stdout, flush=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stdout}")
    return result.stdout


def command_output(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd)
    return result.returncode, result.stdout


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_download(path, expected_sha256):
    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(f"Download produced an empty file: {path}")
    if expected_sha256:
        actual = sha256_file(path)
        if actual.lower() != expected_sha256.lower():
            raise RuntimeError(f"SHA-256 mismatch for {path.name}: expected {expected_sha256}, got {actual}")


def download(url, path, expected_sha256=""):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        try:
            verify_download(path, expected_sha256)
            print(f"Already exists and verified: {path}", flush=True)
            return
        except Exception as exc:
            print(f"Cached file failed verification, downloading again: {exc}", flush=True)
            path.unlink(missing_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    print(f"Downloading {url} -> {path}", flush=True)

    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 KataGo-Modal/1.0",
        "Accept": "*/*",
    }
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=120) as response, open(tmp, "wb") as out:
            shutil.copyfileobj(response, out)
    except Exception as exc:
        print(f"Python download failed: {exc!r}", flush=True)
        print("Trying wget fallback...", flush=True)
        tmp.unlink(missing_ok=True)
        run(
            f'wget --tries=3 --timeout=60 --user-agent="Mozilla/5.0 KataGo-Modal/1.0" -O "{tmp}" "{url}"',
            check=True,
        )

    try:
        verify_download(tmp, expected_sha256)
        os.replace(tmp, path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


def _analysis_config_text(server_threads=None, max_batch_size=None, include_benchmark_threads=False):
    server_lines = []
    if server_threads is not None:
        server_lines.append(f"numNNServerThreadsPerModel = {server_threads}")
    device_count = server_threads or 1
    server_lines.extend(
        f"cudaDeviceToUseThread{index} = {GPU_CUDA_DEVICE}"
        for index in range(device_count)
    )
    if max_batch_size is None:
        max_batch_size = max(1, GPU_SEARCH_THREADS * GPU_ANALYSIS_THREADS)
    server_lines.append(f"nnMaxBatchSize = {max_batch_size}")
    backend_config = "\n".join(server_lines)
    benchmark_threads = f"numSearchThreads = {GPU_SEARCH_THREADS * GPU_ANALYSIS_THREADS}\n" if include_benchmark_threads else ""
    analysis_threads = "" if include_benchmark_threads else (
        f"numSearchThreadsPerAnalysisThread = {GPU_SEARCH_THREADS}\n"
        f"numAnalysisThreads = {GPU_ANALYSIS_THREADS}\n"
    )
    return f"""logDir = {KATAGO_DIR}/logs

maxVisits = {GPU_MAX_VISITS}

{benchmark_threads}{analysis_threads}

nnCacheSizePowerOfTwo = {GPU_NN_CACHE_POWER}
{backend_config}
"""


def _measured_gpu_settings(katago_exec):
    identity = "|".join([
        KATAGO_RELEASE_CACHE_KEY, MODAL_GPU, MODEL_SHA256 or MODEL_FILENAME,
        str(GPU_SEARCH_THREADS),
        str(GPU_ANALYSIS_THREADS),
    ])
    key = hashlib.sha256(identity.encode()).hexdigest()[:24]
    cache_path = KATAGO_DIR / "tuning" / f"cuda_{key}.json"
    if AUTO_TUNE_GPU_SETTINGS and cache_path.exists():
        try:
            cached = json.loads(cache_path.read_text())
            if cached.get("schema") == 2:
                print("Using cached KataGo GPU tuning:", cache_path, flush=True)
                return cached.get("server_threads"), cached.get("max_batch_size")
        except Exception as exc:
            print("Ignoring invalid GPU tuning cache:", exc, flush=True)
    if not AUTO_TUNE_GPU_SETTINGS:
        return None, None

    CONFIG_PATH.write_text(_analysis_config_text(include_benchmark_threads=True))
    command = [
        str(katago_exec), "benchmark",
        "-model", str(MODEL_PATH),
        "-config", str(CONFIG_PATH),
        "-threads", str(GPU_SEARCH_THREADS * GPU_ANALYSIS_THREADS),
        "-visits", str(GPU_TUNING_VISITS),
        "-numpositions", str(GPU_TUNING_POSITIONS),
    ]
    print("Validating and tuning this T4/model with KataGo v1.18.2...", flush=True)
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=900,
    )
    print(result.stdout, flush=True)
    if result.returncode != 0:
        raise RuntimeError("KataGo GPU/model benchmark failed, the Modal runtime is not usable.")
    server_threads = 2 if "ADDITIONAL RECOMMENDATION: 2 NN server threads" in result.stdout else 1
    batch_match = re.search(
        r"ADDITIONAL RECOMMENDATION: a smaller batch size.*?nnMaxBatchSize = (\d+)",
        result.stdout, flags=re.DOTALL,
    )
    max_batch_size = int(batch_match.group(1)) if batch_match else None
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(json.dumps({
        "schema": 2, "server_threads": server_threads,
        "max_batch_size": max_batch_size,
    }, indent=2))
    return server_threads, max_batch_size


def ensure_katago_files():
    KATAGO_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    if not KATAGO_BIN.exists() and not KATAGO_EXTRACTED_BIN.exists():
        zip_path = KATAGO_DIR / "katago.zip"
        download(KATAGO_URL, zip_path, KATAGO_SHA256)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(KATAGO_DIR)
        zip_path.unlink(missing_ok=True)

    download(MODEL_URL, MODEL_PATH, MODEL_SHA256)
    if ENABLE_HUMAN_MODEL:
        download(HUMAN_MODEL_URL, HUMAN_MODEL_PATH, HUMAN_MODEL_SHA256)

    CONFIG_PATH.write_text(_analysis_config_text())

    # Prefer the cached AppRun: probing the AppImage first needs FUSE and
    # produces misleading mount errors in Modal containers.
    katago_exec = KATAGO_EXTRACTED_BIN
    if not KATAGO_EXTRACTED_BIN.exists():
        KATAGO_BIN.chmod(0o755)
        print(f'\n$ "{KATAGO_BIN}" version', flush=True)
        code, out = command_output(f'"{KATAGO_BIN}" version')
        print(out, flush=True)
        if code == 0:
            katago_exec = KATAGO_BIN
        else:
            print("KataGo did not run directly. Trying AppImage extraction fallback...", flush=True)
            run(f'"{KATAGO_BIN}" --appimage-extract', check=True, cwd=KATAGO_DIR)

    if katago_exec == KATAGO_EXTRACTED_BIN:
        if not KATAGO_EXTRACTED_BIN.exists():
            raise RuntimeError(f"AppImage extraction did not create {KATAGO_EXTRACTED_BIN}")
        # Restored caches can lose permissions on the bundled executable too.
        (KATAGO_EXTRACTED_BIN.parent / "usr" / "bin" / "katago").chmod(0o755)
        KATAGO_EXTRACTED_BIN.chmod(0o755)
        out = run(f'"{KATAGO_EXTRACTED_BIN}" version', check=True)
    if not re.search(r"(?:^|[^0-9])1\.18\.2(?:[^0-9]|$)", out):
        raise RuntimeError(f"Expected KataGo 1.18.2, got:\n{out}")

    run("nvidia-smi", check=True)
    measured_servers, measured_batch = _measured_gpu_settings(katago_exec)
    server_threads = GPU_NN_SERVER_THREADS if GPU_NN_SERVER_THREADS is not None else measured_servers
    max_batch_size = GPU_NN_MAX_BATCH_SIZE if GPU_NN_MAX_BATCH_SIZE is not None else measured_batch
    CONFIG_PATH.write_text(_analysis_config_text(server_threads, max_batch_size))
    return katago_exec


function_options = {
    "image": image,
    "volumes": {"/cache": volume},
    "timeout": 60 * 60 * 4,
    "cpu": 2.0,
    "memory": 4096,
    "gpu": MODAL_GPU,
    "max_containers": 1,
    "scaledown_window": 60,
}


@app.function(**function_options)
@modal.concurrent(max_inputs=20)
@modal.asgi_app()
def go():
    from fastapi import FastAPI, WebSocket, WebSocketDisconnect
    from fastapi.responses import PlainTextResponse

    katago_proc = None
    reader_tasks = []
    # Internal IDs are unique per submission, client IDs belong to one connection.
    pending: Dict[str, Tuple[WebSocket, asyncio.Lock, str]] = {}
    client_queries: Dict[WebSocket, Dict[str, str]] = {}
    pending_expected_turns: Dict[str, set] = {}
    pending_completed_turns: Dict[str, set] = {}
    katago_write_lock = asyncio.Lock()
    logs = deque(maxlen=ENGINE_LOG_LIMIT)
    stale_result_ids = set()
    stale_result_order = deque()
    katago_startup_finished = asyncio.Event()

    state = {
        "mode": MODE_LABEL,
        "started_at": time.time(),
        "katago_started": False,
        "katago_ready": False,
        "websocket_clients_total": 0,
        "websocket_clients_active": 0,
        "queries_received": 0,
        "results_sent": 0,
        "errors": 0,
        "warnings": 0,
        "last_error": None,
        "last_query_id": None,
        "last_result_id": None,
        "last_best_move": None,
        "last_visits": None,
        "last_winrate": None,
        "last_score_lead": None,
        "pending_queries": 0,
        "stale_results_dropped": 0,
        "human_model_enabled": ENABLE_HUMAN_MODEL,
    }

    def now_time():
        return time.strftime("%H:%M:%S")

    async def emit(event_type, message, extra=None):
        if event_type == "error":
            state["errors"] += 1
            state["last_error"] = str(message)
        if event_type == "warning":
            state["warnings"] += 1
        state["pending_queries"] = len(pending)
        item = {"ts": time.time(), "time": now_time(), "type": event_type, "message": str(message), "state": dict(state)}
        if extra:
            item.update(extra)
        logs.append(item)
        print(f"[{item['time']}] {event_type}: {message}", flush=True)

    def first_stale_result(query_id):
        if not query_id or query_id in stale_result_ids:
            return False
        stale_result_ids.add(query_id)
        stale_result_order.append(query_id)
        while len(stale_result_order) > 500:
            stale_result_ids.discard(stale_result_order.popleft())
        return True

    def finish_pending_query(query_id):
        target = pending.pop(query_id, None)
        if target:
            websocket, _, client_id = target
            query_ids = client_queries.get(websocket, {})
            if query_ids.get(client_id) == query_id:
                query_ids.pop(client_id, None)
        pending_expected_turns.pop(query_id, None)
        pending_completed_turns.pop(query_id, None)

    async def terminate_pending_query(query_id, reason, retain_until_final=False, turn_numbers=None):
        if query_id not in pending:
            return
        if not retain_until_final:
            finish_pending_query(query_id)
        try:
            terminate = {"id": f"terminate-{query_id}-{time.time_ns()}", "action": "terminate", "terminateId": query_id}
            if turn_numbers is not None:
                terminate["turnNumbers"] = turn_numbers
            await write_to_katago(terminate)
            await emit("query", f"Terminated previous pending query id={query_id}: {reason}", {"query_id": query_id})
        except Exception as exc:
            await emit("error", f"Could not terminate query id={query_id}: {exc!r}")

    async def write_to_katago(payload):
        if katago_proc is None or katago_proc.stdin is None:
            raise RuntimeError("KataGo process is not running")
        if katago_proc.returncode is not None:
            raise RuntimeError(f"KataGo process exited with code {katago_proc.returncode}")
        async with katago_write_lock:
            katago_proc.stdin.write((json.dumps(payload) + "\n").encode("utf-8"))
            await katago_proc.stdin.drain()

    async def start_katago():
        nonlocal katago_proc
        try:
            await emit("system", "Preparing KataGo files in Modal volume...")
            katago_exec = await asyncio.to_thread(ensure_katago_files)
            try:
                await asyncio.to_thread(volume.commit)
            except Exception as exc:
                await emit("warning", f"Volume commit warning: {exc!r}")

            args = [str(katago_exec), "analysis", "-model", str(MODEL_PATH), "-config", str(CONFIG_PATH)]
            if ENABLE_HUMAN_MODEL:
                args.extend(["-human-model", str(HUMAN_MODEL_PATH)])
            else:
                await emit("warning", "Human SL model disabled, humanSLProfile queries will fail.")

            await emit("system", "Launch command: " + " ".join(args))
            katago_proc = await asyncio.create_subprocess_exec(
                *args,
                stdin=asyncio.subprocess.PIPE,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
                limit=SUBPROCESS_BUFFER_MB * 1024 * 1024,
            )
            state["katago_started"] = True
            reader_tasks.extend([
                asyncio.create_task(read_stdout()),
                asyncio.create_task(read_stderr()),
            ])
            await emit("system", "KataGo process started")
        except Exception as exc:
            await emit("error", f"KataGo startup failed: {exc!r}")
            katago_startup_finished.set()

    async def read_stdout():
        while True:
            line = await katago_proc.stdout.readline()
            if not line:
                await emit("system", "KataGo stdout closed")
                break
            text = line.decode("utf-8", errors="ignore").strip()
            if not text:
                continue
            try:
                data = json.loads(text)
                query_id = data.get("id")
                terminate_id = data.get("terminateId")
                if terminate_id:
                    await emit("result", f"Termination acknowledged id={terminate_id}")
                    continue
                if query_id is None and (data.get("error") or data.get("warning") or data.get("noResults")):
                    detail = data.get("error") or data.get("warning") or "KataGo returned no results"
                    level = "error" if data.get("error") or data.get("noResults") else "warning"
                    await emit(level, f"Unscoped KataGo {level}: {detail}")
                    delivered = set()
                    for websocket, send_lock, _ in list(pending.values()):
                        if id(websocket) in delivered:
                            continue
                        delivered.add(id(websocket))
                        try:
                            async with send_lock:
                                await websocket.send_text(json.dumps(data))
                        except Exception as exc:
                            await emit("warning", f"Could not forward unscoped KataGo error: {exc!r}")
                    continue
                target = pending.get(query_id)
                if not target:
                    state["stale_results_dropped"] += 1
                    if first_stale_result(query_id):
                        await emit("stale", f"Dropped stale result id={query_id}, client already cancelled or replaced this query")
                    continue
                websocket, send_lock, client_id = target
                async with send_lock:
                    await websocket.send_text(json.dumps(dict(data, id=client_id)))
                state["results_sent"] += 1
                state["last_result_id"] = query_id
                move_infos = data.get("moveInfos", [])
                if move_infos:
                    state["last_best_move"] = move_infos[0].get("move")
                    state["last_visits"] = move_infos[0].get("visits")
                    state["last_winrate"] = move_infos[0].get("winrate")
                    state["last_score_lead"] = move_infos[0].get("scoreLead")
                await emit("result", f"Sent result id={query_id}, best={state['last_best_move']}, visits={state['last_visits']}")
                if data.get("error"):
                    finish_pending_query(query_id)
                elif data.get("noResults") or data.get("isDuringSearch") is False:
                    expected = pending_expected_turns.get(query_id, set())
                    if not expected:
                        finish_pending_query(query_id)
                    else:
                        turn_number = data.get("turnNumber")
                        if turn_number is not None:
                            pending_completed_turns.setdefault(query_id, set()).add(int(turn_number))
                        if expected.issubset(pending_completed_turns.get(query_id, set())):
                            finish_pending_query(query_id)
            except Exception as exc:
                await emit("error", f"Error forwarding KataGo output: {exc!r}")

    async def read_stderr():
        while True:
            line = await katago_proc.stderr.readline()
            if not line:
                if not state["katago_ready"]:
                    await emit("error", "KataGo stderr closed before the ready marker")
                    katago_startup_finished.set()
                break
            msg = line.decode("utf-8", errors="ignore").rstrip()
            lower = msg.lower()
            if "started, ready to begin handling requests" in lower:
                state["katago_ready"] = True
                katago_startup_finished.set()
                await emit("katago", "KataGo ready")
            elif "warning" in lower:
                await emit("warning", msg)
            elif "error" in lower or "failed" in lower:
                await emit("error", msg)
            elif "tuning" in lower or "calls/sec" in lower or "done tuning" in lower:
                await emit("tuning", msg)
            else:
                print("KataGo:", msg, flush=True)

    @asynccontextmanager
    async def lifespan(_app):
        startup_task = asyncio.create_task(start_katago())
        try:
            await emit("system", "Status service is online. KataGo is starting in the background.")
            yield
        finally:
            # Stop preparation before it can launch an engine after shutdown.
            startup_task.cancel()
            await asyncio.gather(startup_task, return_exceptions=True)
            try:
                if katago_proc and katago_proc.returncode is None:
                    try:
                        katago_proc.terminate()
                    except ProcessLookupError:
                        pass
                    try:
                        await asyncio.wait_for(katago_proc.wait(), timeout=5)
                    except asyncio.TimeoutError:
                        katago_proc.kill()
                        await katago_proc.wait()
            finally:
                for task in reader_tasks:
                    task.cancel()
                await asyncio.gather(*reader_tasks, return_exceptions=True)
                state["katago_ready"] = False
                state["katago_started"] = False
                katago_startup_finished.set()

    web_app = FastAPI(lifespan=lifespan)

    @web_app.get("/")
    async def home():
        return {"ok": True, "message": "KataGo Modal server running", "websocket": KATAGO_WS_PATH, "status": "/status", "logs": "/logs"}

    @web_app.get("/status")
    async def status():
        state["pending_queries"] = len(pending)
        return {"ok": True, "state": state, "recent_logs": list(logs)[-50:]}

    @web_app.get("/logs", response_class=PlainTextResponse)
    async def get_logs(type: str = "all", q: str = ""):
        rows = list(logs)
        if type and type != "all":
            wanted = {t.strip() for t in type.split(",") if t.strip()}
            rows = [item for item in rows if item.get("type") in wanted]
        if q:
            q_lower = q.lower()
            rows = [item for item in rows if q_lower in item.get("message", "").lower()]
        return "\n".join(f"[{item['time']}] {item['type']}: {item['message']}" for item in rows)

    async def handle_ws(websocket: WebSocket):
        await websocket.accept()
        send_lock = asyncio.Lock()
        query_ids = {}
        client_queries[websocket] = query_ids
        state["websocket_clients_total"] += 1
        state["websocket_clients_active"] += 1
        await emit("client", f"connected active={state['websocket_clients_active']}")
        try:
            await katago_startup_finished.wait()
            if not state["katago_ready"]:
                reason = state["last_error"] or "KataGo failed to become ready"
                await websocket.close(code=1011, reason=str(reason)[:120])
                return
            while True:
                message = await websocket.receive_text()
                query = json.loads(message)
                if query.get("action") == "terminate":
                    target_id = query_ids.get(query.get("terminateId"))
                    if target_id:
                        await terminate_pending_query(target_id, "client requested termination", retain_until_final=True, turn_numbers=query.get("turnNumbers"))
                    async with send_lock:
                        await websocket.send_text(json.dumps(query))
                    continue
                if query.get("action") == "terminate_all":
                    # A client's stop-all must not cancel another connection's work.
                    for target_id in list(query_ids.values()):
                        await terminate_pending_query(target_id, "client requested termination", retain_until_final=True, turn_numbers=query.get("turnNumbers"))
                    async with send_lock:
                        await websocket.send_text(json.dumps(query))
                    continue
                client_id = query.get("id") or str(uuid.uuid4())
                previous_id = query_ids.get(client_id)
                if previous_id:
                    await terminate_pending_query(previous_id, "replaced by newer query with the same id")
                query_id = str(uuid.uuid4())
                query["id"] = query_id
                query_ids[client_id] = query_id
                pending[query_id] = (websocket, send_lock, client_id)
                pending_expected_turns[query_id] = {int(turn) for turn in (query.get("analyzeTurns") or [])}
                pending_completed_turns[query_id] = set()
                state["queries_received"] += 1
                state["last_query_id"] = query_id
                await emit("query", f"Received query id={query_id}, moves={len(query.get('moves', []))}, maxVisits={query.get('maxVisits')}")
                await write_to_katago(query)
        except WebSocketDisconnect:
            await emit("client", "disconnected")
        except Exception as exc:
            await emit("error", f"WebSocket error: {exc!r}")
        finally:
            state["websocket_clients_active"] = max(0, state["websocket_clients_active"] - 1)
            dead = [qid for qid, item in pending.items() if item[0] is websocket]
            for qid in dead:
                finish_pending_query(qid)
                try:
                    await write_to_katago({"id": f"terminate-{qid}-{time.time_ns()}", "action": "terminate", "terminateId": qid})
                except Exception:
                    pass
            client_queries.pop(websocket, None)
            await emit("client", f"cleanup active={state['websocket_clients_active']}")

    @web_app.websocket("/")
    async def websocket_root(websocket: WebSocket):
        await handle_ws(websocket)

    @web_app.websocket(KATAGO_WS_PATH)
    async def websocket_katago(websocket: WebSocket):
        await handle_ws(websocket)

    return web_app


In [ ]:
%%swhub_action deploy
# @title Register deploy (runs only from the panel)
import os, queue, re, subprocess, threading, time, urllib.request
from pathlib import Path

APP_NAME_OVERRIDE = ""  # optional, leave blank to read the app name from modal_katago_server.py
STARTUP_LOG_TIMEOUT_SECONDS = 900
KEEP_STREAMING_AFTER_READY = False  # set True if you want logs to keep running after URLs print
TRIGGER_APP_AFTER_DEPLOY = True

if "modal_command" not in globals():
    raise RuntimeError("Click Select workspace before deploying.")

env = os.environ.copy()
env["NO_COLOR"] = "1"
env["TERM"] = "dumb"

def stream_command(cmd):
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    return proc.wait(), lines

def trigger_app(base_url):
    # Modal starts the container on first web request, not merely on deploy.
    url = base_url.rstrip("/") + "/status"
    print(f"\nTriggering Modal cold start by opening: {url}")
    for attempt in range(1, 4):
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                print(f"Status endpoint responded with HTTP {response.status}")
                return
        except Exception as exc:
            print(f"Status trigger attempt {attempt} still starting: {exc!r}")
            time.sleep(5)

def detect_app_name(script_text):
    app_name_match = re.search(r"^APP_NAME\s*=\s*['\"]([^'\"]+)['\"]", script_text, re.MULTILINE)
    if app_name_match:
        return app_name_match.group(1)
    modal_app_match = re.search(r"modal\.App\(['\"]([^'\"]+)['\"]\)", script_text)
    if modal_app_match:
        return modal_app_match.group(1)
    return "katago"

def start_log_reader(app_name):
    proc = subprocess.Popen(
        modal_command("app", "logs", app_name, "--follow"),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    q = queue.Queue()

    def reader():
        for line in proc.stdout:
            q.put(line)
        q.put(None)

    threading.Thread(target=reader, daemon=True).start()
    return proc, q

script_path = Path.cwd() / "modal_katago_server.py"
if not script_path.exists():
    raise FileNotFoundError(
        f"{script_path} does not exist. Click Deploy to prepare the app first, "
        "it should start with %%writefile modal_katago_server.py."
    )

script_text = script_path.read_text()
APP_NAME = APP_NAME_OVERRIDE.strip() or detect_app_name(script_text)

print("Deploying Modal app...")
print("Using app file:", script_path)
print("Using Modal app name for logs:", APP_NAME)
return_code, deploy_lines = stream_command(modal_command("deploy", str(script_path)))
if return_code != 0:
    raise RuntimeError(f"modal deploy failed with exit code {return_code}")

modal_urls = []
for line in deploy_lines:
    for url in re.findall(r"https://[^\s]+?\.modal\.run", line):
        clean = url.rstrip(".,)];'")
        if clean not in modal_urls:
            modal_urls.append(clean)

base = modal_urls[-1].rstrip("/") if modal_urls else ""
if not base:
    print("\nDeploy finished, but I could not auto-detect the Modal URL.")
    print("Look above for a https://...modal.run URL, then convert it to wss://.../katago.")

print("\n\nLIVE MODAL STARTUP LOGS")
print("Waiting for KataGo startup. This is the Modal equivalent of the Colab live log tail.")
log_proc, log_queue = start_log_reader(APP_NAME)

if TRIGGER_APP_AFTER_DEPLOY and base:
    threading.Thread(target=trigger_app, args=(base,), daemon=True).start()

ready = False
failed = False
start_time = time.time()
try:
    while True:
        if time.time() - start_time > STARTUP_LOG_TIMEOUT_SECONDS:
            print(f"\nStartup log wait timed out after {STARTUP_LOG_TIMEOUT_SECONDS}s.")
            break
        try:
            line = log_queue.get(timeout=1)
        except queue.Empty:
            continue
        if line is None:
            print("\nModal log stream ended.")
            break
        print(line, end="")
        lower = line.lower()
        if "katago ready" in lower or "started, ready to begin handling requests" in lower:
            ready = True
            if not KEEP_STREAMING_AFTER_READY:
                break
        if "katago startup failed" in lower:
            failed = True
            if not KEEP_STREAMING_AFTER_READY:
                break
except KeyboardInterrupt:
    print("\nStopped live log streaming.")
finally:
    if not KEEP_STREAMING_AFTER_READY:
        log_proc.terminate()

print("\n\nSETUP COMPLETE" if ready else "\n\nSETUP STOPPED BEFORE READY")
if failed:
    print("A startup error appeared in the logs above. Fix that, then click Deploy again.")

if base:
    print("\nFinal SWHub WebSocket URL:")
    print(base.replace("https://", "wss://").replace("http://", "ws://") + "/katago")
    print("\nStatus URL:")
    print(base + "/status")
else:
    print("\nNo Modal URL was auto-detected from deploy output.")


In [ ]:
%%swhub_action stop
# @title Register stop (runs only from the panel)
import subprocess
# subprocess.run(modal_command("app", "logs", "KataGo-Analyse", "--follow"), check=True)
subprocess.run(modal_command("app", "stop", "KataGo-Analyse"), check=True)


## Troubleshooting

| What happened? | What to do |
| --- | --- |
| Modal asks for a payment method | Add a valid payment method in Modal's billing settings. GPU use requires one even with free credits. |
| Your workspace is missing or incorrect | Click Login again, choose the intended workspace in the browser, then click Select workspace. |
| Login expired | Click Login and Select workspace again, then Deploy. |
| Credits remain but the service stopped | Check the workspace usage budget and unlocked credits in Modal's dashboard. |
| The GPU is unavailable | Retry later. This notebook requests a T4. |
| Colab restarted | Run the top Open control panel block again, then use Login and Select workspace before deploying again. Your deployed app is still on Modal unless it was stopped. |
| Deployment is blocked by an authentication rule | Ask the workspace owner about the rule. This setup needs an environment that permits its connection method. Adding the contributor's optional password does not satisfy Modal's separate proxy-authentication rule. |

<details>
<summary>Optional: recover from a Colab runtime update</summary>

Use **Runtime > Change runtime type > Runtime Version** to try an available earlier version, such as `2026.07`, then rerun setup. Keep a CPU runtime for these Modal setup notebooks. Older versions are temporary options. [Google's runtime help](https://research.google.com/colaboratory/runtime-version-faq.html)

</details>

If analysis is still starting, open the printed `/status` or `/logs` address and wait for the downloads and GPU tuning to finish.
